In [0]:
from pyspark.sql.functions import (
    col, count, avg, sum as spark_sum, when, round as spark_round
)

In [0]:
CATALOG = "flightdata"

In [0]:

silver_table = f"{CATALOG}.silver.silver_flights"
gold_delay_table = f"{CATALOG}.gold.gold_delay_by_airline_airport"
gold_volume_table = f"{CATALOG}.gold.gold_daily_flight_volume"
gold_ontime_table = f"{CATALOG}.gold.gold_on_time_performance"

ON_TIME_THRESHOLD_MINUTES = 15 

In [0]:
silverFlightDF = spark.table(silver_table)

###  DELAY PERFORMANCE BY AIRLINE / AIRPORT

In [0]:
gold_delay_by_airline_airport = (silverFlightDF
    .groupBy("airline_name", "departure_iata", "arrival_iata")
    .agg(
        count("*").alias("flight_count"),
        spark_round(avg("departure_delay"), 1).alias("avg_departure_delay_min"),
        spark_round(avg("arrival_delay"), 1).alias("avg_arrival_delay_min"),
    )
    .orderBy(col("avg_departure_delay_min").desc())
)

(gold_delay_by_airline_airport
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_delay_table))


### DAILY FLIGHT VOLUME SUMMARY

In [0]:
gold_daily_flight_volume = (silverFlightDF
    .groupBy("flight_date", "departure_iata")
    .agg(count("*").alias("flight_count"))
    .orderBy("flight_date", col("flight_count").desc())
)

(gold_daily_flight_volume
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("flight_date")
    .saveAsTable(gold_volume_table))

###  ON-TIME PERFORMANCE (based on reported delay minutes)

In [0]:
gold_on_time_performance = (silverFlightDF
    .withColumn(
        "is_on_time",
        when(col("departure_delay") <= ON_TIME_THRESHOLD_MINUTES, 1).otherwise(0)
    )
    .groupBy("airline_name")
    .agg(
        count("*").alias("total_flights"),
        spark_sum("is_on_time").alias("on_time_flights"),
        spark_round(
            (spark_sum("is_on_time") / count("*")) * 100, 1
        ).alias("on_time_pct"),
    )
    .orderBy(col("on_time_pct").desc())
)

(gold_on_time_performance
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_ontime_table))


In [0]:
display(spark.table(gold_delay_table))
display(spark.table(gold_volume_table))
display(spark.table(gold_ontime_table))